In [ ]:
STREAM1_MODEL_WEIGHTS_URL = ""  # compatibility: empty keeps repo stream1_weights
STREAM1_MODEL_TESTS = [
    {"name": "repo_output24", "weights": ""},
]

BEAM_WIDTH = 2**24
START_PUZZLE_ID = 0
PUZZLE_COUNT = 1
DEPTH_LIMIT = 10
RUN_TIMEOUT_SEC = 0

GITHUB_REPO_URL = "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git"
GITHUB_BRANCH = "main"
CUDA_ARCHITECTURES = "75"  # Kaggle 2xT4

TORCHRUN_NNODES = 1
TORCHRUN_NPROC_PER_NODE = 2
TORCHRUN_NODE_RANK = 0
TORCHRUN_RDZV_BACKEND = "c10d"
TORCHRUN_RDZV_ENDPOINT = "127.0.0.1:29500"
TORCHRUN_RDZV_ID = "beam_torchrun_p0_d10"

ENABLE_DEBUG = True
ENABLE_DEPTH_LOGS = True
ENABLE_DEBUG_LOGS = False
DEBUG_STREAM_TIMING = False
DEBUG_INFERENCE_TRACE = False
DEBUG_PATH_TRACE = False
DEBUG_FINAL_VALIDATE = False
DEBUG_FINAL_EXCHANGE_TRACE = False
DEBUG_FINAL_HISTOGRAM_TRACE = False
DEBUG_STREAM4_HISTOGRAM_TRACE = False
DEBUG_DEPTH_FLOW_TRACE = False
DEBUG_PIPELINE_STATS = False

DEPTH_LOG_EVERY = 1
PUZZLE_LOG_EVERY = 1
HISTORY_MODE = "ram"
HISTORY_SLOT_COUNT = 2
HISTORY_WORKERS = 1
HISTORY_RAM_BYTES = 28 * 1024**3
HISTORY_DISK_BYTES = 54 * 1024**3
HISTORY_DISK_PATH = "/tmp/beam_history_arena"
SOLVED_NEIGHBORHOOD_RADIUS = 4
SOLVED_NEIGHBORHOOD_MAX_ENTRIES = 0
STREAM2_SUFFIX_RADIUS = 0
STREAM2_SUFFIX_BACKEND = "composed_permutations"
STREAM2_SUFFIX_MAX_COUNT = 0
STOP_ON_FAILURE = False
LIVE_LOG_RANKS = [0]
RUN_STREAM_BENCHMARK = False

RUNTIME_CONFIG_MODE = "manual"
SHARD_BUFFER_COUNT = 2
STREAM4_BATCH_ALIGNMENT = 1024
SHARD_CAPACITY_SCALE_PPM = 1050000
GLOBAL_SPILL_CAPACITY = 0
STREAM5_RECV_CAPACITY_SCALE_PPM = 1000000
GPU_HEADROOM_BYTES = 224 * 1024**2
B_MICRO = 2048
STREAM1_CONCURRENCY = 4
STREAM3_RING_SLOTS = 4
SHARD_COUNT = 8
STREAM4_ACTIVE_SORT_SLOTS = 4
STREAM4_BATCH_CANDIDATES = 196608
STREAM4_TRIGGER_CANDIDATES = 196608

SWEEP_CONFIGS = [
    {
        "name": "repo_output24_sh16_slot4_b196_t393",
        "MODEL_TEST": "repo_output24",
        "HISTORY_MODE": "static_hybrid",
        "B_MICRO": 2048,
        "STREAM1_CONCURRENCY": 4,
        "STREAM3_RING_SLOTS": 4,
        "SHARD_COUNT": 16,
        "STREAM4_BATCH_CANDIDATES": 196608,
        "STREAM4_TRIGGER_CANDIDATES": 393216,
        "STREAM4_ACTIVE_SORT_SLOTS": 4,
    },
]


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import urllib.request

WORK_DIR = Path('/kaggle/working')
TMP_DIR = Path('/tmp')
REPO_DIR = TMP_DIR / 'beam_solver_t4_torchrun'
CUTLASS_DIR = TMP_DIR / 'cutlass'
BUILD_DIR = TMP_DIR / 'beam_build_t4_torchrun'

def run_checked(cmd, cwd=None, env=None):
    print('+ ' + ' '.join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True)

def safe_name(text):
    return ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in str(text))

def resolve_weight_path(ref):
    path = Path(ref)
    if path.is_file():
        return path
    if path.is_dir():
        matches = sorted(path.rglob('*.pth'))
        if matches:
            return matches[0]
    raise FileNotFoundError(f'no .pth file found for {ref}')

for path in (REPO_DIR, BUILD_DIR):
    if path.exists():
        shutil.rmtree(path)

run_checked(['git', 'clone', '--branch', GITHUB_BRANCH, '--depth', '1', GITHUB_REPO_URL, REPO_DIR])

STREAM1_WEIGHT_DIRS = {}
for model in STREAM1_MODEL_TESTS:
    name = model['name']
    ref = str(model.get('weights', '')).strip()
    if not ref:
        STREAM1_WEIGHT_DIRS[name] = REPO_DIR / 'stream1_weights'
        print(f'stream1_model_weights_source=repo model={name} path={STREAM1_WEIGHT_DIRS[name]}')
        continue
    source_path = TMP_DIR / f'stream1_model_{safe_name(name)}.pth'
    if ref.startswith('/'):
        shutil.copyfile(resolve_weight_path(ref), source_path)
    else:
        urllib.request.urlretrieve(ref, source_path)
    out_dir = TMP_DIR / f'stream1_weights_{safe_name(name)}'
    if out_dir.exists():
        shutil.rmtree(out_dir)
    run_checked(['python3', REPO_DIR / 'tools' / 'export_stream1_mlp.py', '--weights', source_path, '--out', out_dir], cwd=REPO_DIR)
    STREAM1_WEIGHT_DIRS[name] = out_dir
    print(f'stream1_model_weights_exported model={name} out={out_dir}')

if not (CUTLASS_DIR / 'include').exists():
    if CUTLASS_DIR.exists():
        shutil.rmtree(CUTLASS_DIR)
    run_checked(['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/cutlass.git', CUTLASS_DIR])

run_checked([
    'cmake', '-S', REPO_DIR, '-B', BUILD_DIR, '-GNinja',
    '-DCMAKE_BUILD_TYPE=Release',
    f'-DBEAM_CUDA_ARCHITECTURES={CUDA_ARCHITECTURES}',
    f'-DCUTLASS_DIR={CUTLASS_DIR}',
    f'-DBEAM_ENABLE_DEBUG={"ON" if ENABLE_DEBUG else "OFF"}',
    f'-DBEAM_ENABLE_DEPTH_LOGS={"ON" if ENABLE_DEPTH_LOGS else "OFF"}',
    f'-DBEAM_ENABLE_DEBUG_LOGS={"ON" if ENABLE_DEBUG_LOGS else "OFF"}',
    f'-DBEAM_DEBUG_STREAM_TIMING={"ON" if DEBUG_STREAM_TIMING else "OFF"}',
    f'-DBEAM_DEBUG_INFERENCE_TRACE={"ON" if DEBUG_INFERENCE_TRACE else "OFF"}',
    f'-DBEAM_DEBUG_PATH_TRACE={"ON" if DEBUG_PATH_TRACE else "OFF"}',
    f'-DBEAM_DEBUG_FINAL_VALIDATE={"ON" if DEBUG_FINAL_VALIDATE else "OFF"}',
    f'-DBEAM_DEBUG_FINAL_EXCHANGE_TRACE={"ON" if DEBUG_FINAL_EXCHANGE_TRACE else "OFF"}',
    f'-DBEAM_DEBUG_FINAL_HISTOGRAM_TRACE={"ON" if DEBUG_FINAL_HISTOGRAM_TRACE else "OFF"}',
    f'-DBEAM_DEBUG_STREAM4_HISTOGRAM_TRACE={"ON" if DEBUG_STREAM4_HISTOGRAM_TRACE else "OFF"}',
    f'-DBEAM_DEBUG_DEPTH_FLOW_TRACE={"ON" if DEBUG_DEPTH_FLOW_TRACE else "OFF"}',
    f'-DBEAM_DEBUG_PIPELINE_STATS={"ON" if DEBUG_PIPELINE_STATS else "OFF"}',
], cwd=REPO_DIR)
run_checked(['cmake', '--build', BUILD_DIR, '--target', 'production_runner', '-j', '2'])


In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

def torchrun_world_size():
    nnodes = int(TORCHRUN_NNODES)
    nproc_per_node = int(TORCHRUN_NPROC_PER_NODE)
    if nnodes <= 0 or nproc_per_node <= 0:
        raise ValueError(f'invalid torchrun shape: nnodes={nnodes} nproc_per_node={nproc_per_node}')
    return nnodes * nproc_per_node

def cleanup_path(path):
    path = Path(path)
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()

def should_print_live(line):
    low = line.lower()
    return (
        'runner_phase=' in line or
        'candidate_history_' in line or
        'history_' in line or
        line.startswith('depth ') or
        'puzzle_' in line or
        'error' in low or
        'exception' in low or
        'terminate called' in low or
        'what():' in low or
        'childfailed' in low or
        'traceback' in low or
        'RUN_' in line
    )

def cfg_get(cfg, key, default):
    return cfg[key] if key in cfg else default

def round_up(value, alignment):
    return ((int(value) + int(alignment) - 1) // int(alignment)) * int(alignment)

def derived_shard_capacity_candidates(cfg):
    world_size = torchrun_world_size()
    shard_count = int(cfg_get(cfg, 'SHARD_COUNT', SHARD_COUNT))
    alignment = int(STREAM4_BATCH_ALIGNMENT)
    beam_alignment = world_size * shard_count * alignment
    global_beam_effective = round_up(BEAM_WIDTH, beam_alignment)
    local_beam_width = global_beam_effective // world_size
    logical_shard_size = (local_beam_width + shard_count - 1) // shard_count
    capacity_raw = (logical_shard_size * int(SHARD_CAPACITY_SCALE_PPM) + 999999) // 1000000
    return round_up(capacity_raw, alignment)

def env_for_config(cfg, puzzle_id):
    model_name = cfg_get(cfg, 'MODEL_TEST', STREAM1_MODEL_TESTS[0]['name'])
    weight_dir = STREAM1_WEIGHT_DIRS[model_name]
    target_weights = REPO_DIR / 'stream1_weights'
    if target_weights.exists() and target_weights.resolve() != Path(weight_dir).resolve():
        shutil.rmtree(target_weights)
    if not target_weights.exists():
        shutil.copytree(weight_dir, target_weights)

    history_path = Path(HISTORY_DISK_PATH) / f'torchrun_p{puzzle_id}_{safe_name(cfg_get(cfg, "name", "config"))}'
    cleanup_path(history_path)
    history_path.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env.update({
        'BEAM_RUNTIME_CONFIG_MODE': RUNTIME_CONFIG_MODE,
        'BEAM_SHARD_COUNT': str(cfg_get(cfg, 'SHARD_COUNT', SHARD_COUNT)),
        'BEAM_SHARD_CAPACITY_CANDIDATES': str(derived_shard_capacity_candidates(cfg)),
        'BEAM_SHARD_CAPACITY_SCALE_PPM': str(SHARD_CAPACITY_SCALE_PPM),
        'BEAM_SHARD_BUFFER_COUNT': str(SHARD_BUFFER_COUNT),
        'BEAM_GLOBAL_SPILL_CAPACITY': str(GLOBAL_SPILL_CAPACITY),
        'BEAM_STREAM5_RECV_CAPACITY_SCALE_PPM': str(STREAM5_RECV_CAPACITY_SCALE_PPM),
        'BEAM_GPU_HEADROOM_BYTES': str(GPU_HEADROOM_BYTES),
        'BEAM_B_MICRO': str(cfg_get(cfg, 'B_MICRO', B_MICRO)),
        'BEAM_STREAM1_CONCURRENCY': str(cfg_get(cfg, 'STREAM1_CONCURRENCY', STREAM1_CONCURRENCY)),
        'BEAM_STREAM3_RING_SLOTS': str(cfg_get(cfg, 'STREAM3_RING_SLOTS', STREAM3_RING_SLOTS)),
        'BEAM_STREAM4_BATCH_CANDIDATES': str(cfg_get(cfg, 'STREAM4_BATCH_CANDIDATES', STREAM4_BATCH_CANDIDATES)),
        'BEAM_STREAM4_TRIGGER_CANDIDATES': str(cfg_get(cfg, 'STREAM4_TRIGGER_CANDIDATES', STREAM4_TRIGGER_CANDIDATES)),
        'BEAM_STREAM4_ACTIVE_SORT_SLOTS': str(cfg_get(cfg, 'STREAM4_ACTIVE_SORT_SLOTS', STREAM4_ACTIVE_SORT_SLOTS)),
        'BEAM_HISTORY_MODE': str(cfg_get(cfg, 'HISTORY_MODE', HISTORY_MODE)),
        'BEAM_HISTORY_RAM_BYTES': str(HISTORY_RAM_BYTES),
        'BEAM_HISTORY_DISK_BYTES': str(HISTORY_DISK_BYTES),
        'BEAM_HISTORY_DISK_PATH': str(history_path),
        'BEAM_HISTORY_SLOT_COUNT': str(HISTORY_SLOT_COUNT),
        'BEAM_HISTORY_WORKERS': str(HISTORY_WORKERS),
        'BEAM_SOLVED_NEIGHBORHOOD_RADIUS': str(SOLVED_NEIGHBORHOOD_RADIUS),
        'BEAM_SOLVED_NEIGHBORHOOD_MAX_ENTRIES': str(SOLVED_NEIGHBORHOOD_MAX_ENTRIES),
        'BEAM_STREAM2_SUFFIX_RADIUS': str(STREAM2_SUFFIX_RADIUS),
        'BEAM_STREAM2_SUFFIX_BACKEND': str(STREAM2_SUFFIX_BACKEND),
        'BEAM_STREAM2_SUFFIX_MAX_COUNT': str(STREAM2_SUFFIX_MAX_COUNT),
        'BEAM_DEPTH_LOG_EVERY': str(DEPTH_LOG_EVERY),
    })
    env.pop('WORLD_SIZE', None)
    env.pop('RANK', None)
    env.pop('LOCAL_RANK', None)
    return env, history_path

def run_one(puzzle_id, cfg):
    world_size = torchrun_world_size()
    cfg_name = cfg_get(cfg, 'name', 'config')
    env, history_path = env_for_config(cfg, puzzle_id)
    log_path = WORK_DIR / f'torchrun_{safe_name(cfg_name)}_p{puzzle_id}.log'
    cmd = [
        sys.executable, '-m', 'torch.distributed.run',
        '--no-python',
        f'--nnodes={int(TORCHRUN_NNODES)}',
        f'--nproc-per-node={int(TORCHRUN_NPROC_PER_NODE)}',
        f'--node-rank={int(TORCHRUN_NODE_RANK)}',
        f'--rdzv-backend={TORCHRUN_RDZV_BACKEND}',
        f'--rdzv-endpoint={TORCHRUN_RDZV_ENDPOINT}',
        f'--rdzv-id={TORCHRUN_RDZV_ID}',
        str(BUILD_DIR / 'production_runner'), str(puzzle_id), str(DEPTH_LIMIT), str(BEAM_WIDTH),
    ]
    print('RUN_T4_TORCHRUN_START', flush=True)
    print('cmd=', ' '.join(map(str, cmd)), flush=True)
    print('config=', cfg_name, flush=True)
    print('world_size_from_torchrun_shape=', world_size, flush=True)
    print('global_beam=', BEAM_WIDTH, flush=True)
    print('history_path=', history_path, flush=True)
    print('log_path=', log_path, flush=True)
    start = time.time()
    with log_path.open('w', buffering=1, encoding='utf-8') as log:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            log.write(line)
            if should_print_live(line):
                print(line, end='', flush=True)
        rc = proc.wait()
    elapsed = time.time() - start
    print(f'RUN_T4_TORCHRUN_RC {rc} seconds={elapsed:.3f}', flush=True)
    cleanup_path(history_path)
    return rc

results = []
for cfg in SWEEP_CONFIGS:
    for offset in range(PUZZLE_COUNT):
        puzzle_id = START_PUZZLE_ID + offset
        rc = run_one(puzzle_id, cfg)
        results.append({'config': cfg_get(cfg, 'name', 'config'), 'puzzle_id': puzzle_id, 'return_code': rc})
        if rc != 0 and STOP_ON_FAILURE:
            raise SystemExit(rc)
        if rc != 0:
            raise SystemExit(rc)
results
